# Praktikum 6 — Perbandingan ANNOY, FAISS, dan HNSWLIB pada Dataset Micro Spotify

## Tujuan
Menggunakan dataset Micro Spotify dari Kaggle untuk mencari lagu dengan karakteristik audio yang mirip berdasarkan sembilan fitur numerik, lalu membandingkan:
- **Exact Nearest Neighbors** sebagai acuan (*ground truth*)
- **ANNOY**
- **FAISS IVF-Flat**
- **HNSWLIB**

Dataset sumber: https://www.kaggle.com/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/data

Sesuai instruksi praktikum, pilih **dataset/CSV pertama** dari dua dataset pada halaman Kaggle. Nama berkas pada contoh kode adalah `spotify_songs.csv`.

### Catatan metodologis
1. Dataset harus diunduh dari Kaggle dan diunggah ke Google Colab atau disimpan di Google Drive. Notebook ini tidak menganggap dataset sudah tersedia otomatis.
2. Sembilan fitur distandardisasi dengan `StandardScaler`.
3. Untuk menghindari penghitungan tetangga dirinya sendiri sebagai hasil terbaik, query evaluasi diambil sebagai subset dari data dan hasil tetangga dengan ID query dikeluarkan saat menghitung Recall@10.
4. Exact search untuk seluruh baris pada dataset besar dapat mahal. Notebook memakai sejumlah query evaluasi yang bisa diatur; indeks tetap dibangun menggunakan seluruh data valid.
5. Waktu pembangunan indeks (*build time*) dan waktu pencarian (*query time*) diukur terpisah.
6. Angka waktu dan hasil tetangga baru tersedia setelah notebook dijalankan di runtime Anda.


## 1. Instalasi library

In [ ]:
!pip -q install annoy faiss-cpu hnswlib scikit-learn pandas matplotlib

## 2. Unggah atau hubungkan dataset

**Pilihan A — unggah CSV langsung ke Colab:** jalankan sel berikut, lalu pilih berkas CSV pertama yang diunduh dari Kaggle.

**Pilihan B — Google Drive:** ubah `CSV_PATH` ke lokasi berkas Anda, misalnya `/content/drive/MyDrive/Kuliah/spotify_songs.csv`, lalu set `USE_UPLOAD = False`. Jika menggunakan Drive, jalankan `drive.mount` pada sel yang disediakan.

In [ ]:
from google.colab import files
import os

USE_UPLOAD = True
CSV_PATH = "/content/spotify_songs.csv"

if USE_UPLOAD:
    uploaded = files.upload()
    csv_candidates = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
    if not csv_candidates:
        raise FileNotFoundError("Tidak ada file CSV yang diunggah. Unggah CSV pertama dari dataset Kaggle.")
    # Jika lebih dari satu CSV diunggah, pilih kandidat pertama; sebaiknya unggah hanya CSV yang diminta.
    CSV_PATH = csv_candidates[0]
    print("CSV terpilih:", CSV_PATH)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Path CSV:", CSV_PATH)

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"File tidak ditemukan: {CSV_PATH}. Unggah CSV atau sesuaikan CSV_PATH."
    )


## 3. Baca data, pilih fitur numerik, dan bersihkan data

Kolom yang digunakan sama dengan instruksi: `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, dan `tempo`. Baris dengan nilai fitur kosong atau non-numerik dibuang agar indeks dapat dibangun secara konsisten. ID baris asli dipertahankan sebagai identitas internal untuk interpretasi hasil.

In [ ]:
import pandas as pd
import numpy as np
import time
import gc
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import faiss
import hnswlib

FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]
K = 10
N_QUERY = 1000  # jumlah query untuk evaluasi; turunkan jika runtime/RAM terbatas
RANDOM_SEED = 42

df = pd.read_csv(CSV_PATH)
print("Ukuran awal dataset:", df.shape)
print("Kolom yang tersedia:", df.columns.tolist())

missing_cols = [c for c in FEATURES if c not in df.columns]
if missing_cols:
    raise ValueError(
        "Kolom fitur berikut tidak ditemukan: " + str(missing_cols) +
        ". Pastikan berkas yang dipilih adalah CSV pertama yang sesuai dengan instruksi."
    )

# Konversi ke numerik dan hilangkan baris dengan nilai hilang/tidak valid.
numeric_features = df[FEATURES].apply(pd.to_numeric, errors="coerce")
valid_mask = numeric_features.notna().all(axis=1) & np.isfinite(numeric_features).all(axis=1)
df_valid = df.loc[valid_mask].copy().reset_index(drop=True)
X_raw = numeric_features.loc[valid_mask].to_numpy(dtype=np.float32)

if len(X_raw) <= K:
    raise ValueError(f"Dataset valid hanya memiliki {len(X_raw)} baris; harus lebih dari k={K}.")

# StandardScaler bekerja pada float; setelah itu dipastikan float32 untuk ANN libraries.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw).astype(np.float32)
X_scaled = np.ascontiguousarray(X_scaled)

print(f"Baris valid yang digunakan: {len(X_scaled):,}")
print(f"Jumlah fitur: {X_scaled.shape[1]}")
print("Jumlah baris yang dibuang karena nilai fitur tidak valid:", int((~valid_mask).sum()))
print("Rata-rata fitur terstandardisasi (mendekati 0):", np.round(X_scaled.mean(axis=0), 3))
print("Simpangan baku fitur terstandardisasi (mendekati 1):", np.round(X_scaled.std(axis=0), 3))


## 4. Pilih query evaluasi dan hitung Exact Nearest Neighbors

Indeks dibangun dari semua data valid, tetapi evaluasi dilakukan pada subset query untuk mengendalikan waktu komputasi. Untuk setiap query, hasil dengan ID yang sama dengan query dikeluarkan agar lagu tidak dianggap sebagai tetangga terdekat dirinya sendiri.

Exact search di bawah menghitung seluruh jarak pada dataset untuk setiap query terpilih. Ini menjadi acuan Recall@10 bagi ANNOY, FAISS, dan HNSWLIB.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n_items = X_scaled.shape[0]
n_query = min(N_QUERY, n_items)
query_ids = rng.choice(n_items, size=n_query, replace=False)
X_query = np.ascontiguousarray(X_scaled[query_ids])

def remove_self_and_take_k(ids, query_id, k):
    ids = [int(i) for i in ids if int(i) >= 0 and int(i) != int(query_id)]
    return ids[:k]

def exact_search_for_queries(X, query_ids, k=K, chunk_size=256):
    # Memproses query per batch untuk membatasi pemakaian memori.
    exact_lists = []
    X64 = X.astype(np.float32, copy=False)
    t0 = time.perf_counter()
    for start in range(0, len(query_ids), chunk_size):
        q_ids_batch = query_ids[start:start + chunk_size]
        Q = X64[q_ids_batch]
        # Jarak Euclidean kuadrat; pengurutan sama dengan Euclidean.
        q_norm = np.sum(Q * Q, axis=1, keepdims=True)
        x_norm = np.sum(X64 * X64, axis=1)[None, :]
        d2 = q_norm + x_norm - 2.0 * (Q @ X64.T)
        np.maximum(d2, 0, out=d2)
        for local_i, query_id in enumerate(q_ids_batch):
            d2[local_i, query_id] = np.inf  # jangan kembalikan query itu sendiri
            cand = np.argpartition(d2[local_i], k - 1)[:k]
            cand = cand[np.argsort(d2[local_i, cand])]
            exact_lists.append(cand.astype(int).tolist())
        del d2
    elapsed = time.perf_counter() - t0
    return exact_lists, elapsed

k_eval = min(K, n_items - 1)
exact_ids, exact_query_seconds = exact_search_for_queries(X_scaled, query_ids, k=k_eval)
print(f"Exact search selesai untuk {n_query:,} query dalam {exact_query_seconds:.3f} detik.")
print("Contoh exact neighbors untuk query pertama:", exact_ids[0][:5])


In [ ]:
def recall_at_k(predicted, ground_truth, k=K):
    total_correct = 0
    total = len(ground_truth) * k
    for pred, truth in zip(predicted, ground_truth):
        total_correct += len(set(pred[:k]) & set(truth[:k]))
    return total_correct / total if total else 0.0

def clean_neighbor_rows(rows, query_ids, k=K):
    cleaned = []
    for ids, qid in zip(rows, query_ids):
        cleaned.append(remove_self_and_take_k(ids, qid, k))
    return cleaned

def pad_rows(rows, k=K):
    # Tidak diperlukan untuk recall set-based, tetapi berguna untuk pemeriksaan.
    return [row[:k] for row in rows]


## 5. Benchmark ANNOY

ANNOY membangun beberapa random projection trees. Parameter `n_trees` memengaruhi biaya build dan kualitas pencarian; `search_k` mengendalikan jumlah kandidat yang diperiksa saat query. Waktu build dan query diukur terpisah.

In [ ]:
def run_annoy(X, X_query, query_ids, exact_ids, n_trees=10, search_k=1000, k=K):
    dim = X.shape[1]
    index = AnnoyIndex(dim, "euclidean")

    t0 = time.perf_counter()
    for i, vec in enumerate(X):
        index.add_item(i, vec.tolist())
    index.build(n_trees)
    build_seconds = time.perf_counter() - t0

    t0 = time.perf_counter()
    raw_ids = [
        index.get_nns_by_vector(q.tolist(), k + 1, search_k=search_k)
        for q in X_query
    ]
    query_seconds = time.perf_counter() - t0

    pred_ids = clean_neighbor_rows(raw_ids, query_ids, k)
    recall = recall_at_k(pred_ids, exact_ids, k)
    return {
        "Metode": "ANNOY",
        "Parameter": f"n_trees={n_trees}, search_k={search_k}",
        "Recall@10": recall,
        "Build time (s)": build_seconds,
        "Query time (s)": query_seconds,
        "Query time (ms/query)": query_seconds * 1000 / len(X_query),
        "QPS": len(X_query) / query_seconds if query_seconds else np.inf,
        "Contoh tetangga query pertama": pred_ids[0][:5]
    }, pred_ids

annoy_results = []
annoy_predictions = {}
for search_k in [200, 500, 1000, 2000]:
    print("Menjalankan ANNOY search_k =", search_k)
    row, pred = run_annoy(
        X_scaled, X_query, query_ids, exact_ids,
        n_trees=10, search_k=search_k, k=k_eval
    )
    annoy_results.append(row)
    annoy_predictions[search_k] = pred
    print(f"  Recall@10={row['Recall@10']:.4f}, query={row['Query time (s)']:.3f}s")


## 6. Benchmark HNSWLIB

HNSWLIB membangun graf navigasi bertingkat. Parameter `M` mengendalikan konektivitas graf, `ef_construction` memengaruhi kualitas indeks saat dibangun, dan `ef` memengaruhi jumlah kandidat saat pencarian. Nilai `ef` lebih tinggi biasanya meningkatkan recall dengan tambahan waktu query.

In [ ]:
def run_hnsw(X, X_query, query_ids, exact_ids, ef=200, M=16, ef_construction=200, k=K):
    dim = X.shape[1]
    index = hnswlib.Index(space="l2", dim=dim)

    t0 = time.perf_counter()
    index.init_index(
        max_elements=X.shape[0],
        ef_construction=ef_construction,
        M=M,
        random_seed=RANDOM_SEED
    )
    index.add_items(X, np.arange(X.shape[0]))
    build_seconds = time.perf_counter() - t0

    index.set_ef(max(ef, k + 1))
    t0 = time.perf_counter()
    labels, distances = index.knn_query(X_query, k=min(k + 1, X.shape[0]))
    query_seconds = time.perf_counter() - t0

    pred_ids = clean_neighbor_rows(labels.tolist(), query_ids, k)
    recall = recall_at_k(pred_ids, exact_ids, k)
    row = {
        "Metode": "HNSWLIB",
        "Parameter": f"M={M}, ef_construction={ef_construction}, ef={ef}",
        "Recall@10": recall,
        "Build time (s)": build_seconds,
        "Query time (s)": query_seconds,
        "Query time (ms/query)": query_seconds * 1000 / len(X_query),
        "QPS": len(X_query) / query_seconds if query_seconds else np.inf,
        "Contoh tetangga query pertama": pred_ids[0][:5]
    }
    return row, pred_ids

hnsw_results = []
hnsw_predictions = {}
for ef in [50, 100, 200, 400]:
    print("Menjalankan HNSW ef =", ef)
    row, pred = run_hnsw(
        X_scaled, X_query, query_ids, exact_ids,
        ef=ef, M=16, ef_construction=200, k=k_eval
    )
    hnsw_results.append(row)
    hnsw_predictions[ef] = pred
    print(f"  Recall@10={row['Recall@10']:.4f}, query={row['Query time (s)']:.3f}s")
    gc.collect()


## 7. Benchmark FAISS IVF-Flat

FAISS IVF-Flat membagi vektor ke dalam inverted lists. `nlist` menentukan jumlah cluster dan `nprobe` menentukan jumlah cluster yang diperiksa ketika query dijalankan. Pada eksperimen ini, indeks menggunakan L2 dan mempertahankan vektor penuh (tidak menggunakan Product Quantization), sehingga jarak dalam cluster terpilih dihitung tanpa kompresi lossy.

In [ ]:
def run_faiss_ivf(X, X_query, query_ids, exact_ids, nlist=100, nprobe=10, k=K):
    dim = X.shape[1]
    nlist_actual = min(nlist, max(1, X.shape[0] // 40))
    quantizer = faiss.IndexFlatL2(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist_actual, faiss.METRIC_L2)

    t0 = time.perf_counter()
    index.train(X)
    index.add(X)
    build_seconds = time.perf_counter() - t0

    index.nprobe = min(nprobe, nlist_actual)
    t0 = time.perf_counter()
    distances, labels = index.search(X_query, min(k + 1, X.shape[0]))
    query_seconds = time.perf_counter() - t0

    pred_ids = clean_neighbor_rows(labels.tolist(), query_ids, k)
    recall = recall_at_k(pred_ids, exact_ids, k)
    row = {
        "Metode": "FAISS IVF-Flat",
        "Parameter": f"nlist={nlist_actual}, nprobe={min(nprobe, nlist_actual)}",
        "Recall@10": recall,
        "Build time (s)": build_seconds,
        "Query time (s)": query_seconds,
        "Query time (ms/query)": query_seconds * 1000 / len(X_query),
        "QPS": len(X_query) / query_seconds if query_seconds else np.inf,
        "Contoh tetangga query pertama": pred_ids[0][:5]
    }
    return row, pred_ids

faiss_results = []
faiss_predictions = {}
for nprobe in [1, 5, 10, 20]:
    print("Menjalankan FAISS IVF nprobe =", nprobe)
    row, pred = run_faiss_ivf(
        X_scaled, X_query, query_ids, exact_ids,
        nlist=100, nprobe=nprobe, k=k_eval
    )
    faiss_results.append(row)
    faiss_predictions[nprobe] = pred
    print(f"  Recall@10={row['Recall@10']:.4f}, query={row['Query time (s)']:.3f}s")
    gc.collect()


## 8. Tabel perbandingan hasil

Tabel ini berisi recall, waktu build, waktu query, QPS, dan contoh tetangga untuk setiap konfigurasi. Angka merupakan hasil pengukuran runtime aktual setelah sel-sel benchmark dijalankan.

In [ ]:
all_results = annoy_results + hnsw_results + faiss_results
results_df = pd.DataFrame(all_results)
display(results_df[[
    "Metode", "Parameter", "Recall@10",
    "Build time (s)", "Query time (s)",
    "Query time (ms/query)", "QPS",
    "Contoh tetangga query pertama"
]].sort_values(["Metode", "Query time (ms/query)"]))


## 9. Visualisasi Recall vs QPS

Grafik ini menampilkan trade-off akurasi dan kecepatan. Titik yang lebih ke kanan berarti recall lebih tinggi; titik yang lebih ke atas berarti QPS lebih tinggi. Posisi kanan-atas umumnya lebih diinginkan.

In [ ]:
plt.figure(figsize=(9, 6))
for method in results_df["Metode"].unique():
    subset = results_df[results_df["Metode"] == method]
    plt.plot(
        subset["Recall@10"], subset["QPS"],
        marker="o", linewidth=1.8, label=method
    )
    for _, row in subset.iterrows():
        plt.annotate(
            row["Parameter"].split(",")[-1].strip(),
            (row["Recall@10"], row["QPS"]),
            textcoords="offset points", xytext=(4, 4), fontsize=7
        )

plt.xlabel("Recall@10")
plt.ylabel("Queries per second (QPS)")
plt.title("Praktikum 6 — Micro Spotify: Recall vs QPS")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()


## 10. Tampilkan tetangga dari satu track

Bagian ini menampilkan hasil indeks tetangga untuk query pertama. ID yang ditampilkan adalah indeks baris pada `df_valid`, bukan Spotify track ID asli. Jika CSV memiliki kolom seperti `track_name`, `track`, atau `artists`, notebook mencoba menampilkan metadata tersebut untuk memudahkan interpretasi.

In [ ]:
def show_track_rows(row_ids):
    possible_name_cols = ["track_name", "track", "name", "song_name"]
    possible_artist_cols = ["artists", "artist", "artist_name"]
    name_col = next((c for c in possible_name_cols if c in df_valid.columns), None)
    artist_col = next((c for c in possible_artist_cols if c in df_valid.columns), None)

    records = []
    for rank, rid in enumerate(row_ids, start=1):
        record = {"Rank": rank, "Row index": int(rid)}
        if name_col:
            record["Track"] = df_valid.iloc[int(rid)][name_col]
        if artist_col:
            record["Artist"] = df_valid.iloc[int(rid)][artist_col]
        records.append(record)
    return pd.DataFrame(records)

print("Query pertama berasal dari row index:", int(query_ids[0]))
print("\nExact NN:")
display(show_track_rows(exact_ids[0]))

print("\nContoh hasil ANNOY (search_k=1000):")
display(show_track_rows(annoy_predictions[1000][0]))

print("\nContoh hasil HNSWLIB (ef=200):")
display(show_track_rows(hnsw_predictions[200][0]))

print("\nContoh hasil FAISS IVF (nprobe=10):")
display(show_track_rows(faiss_predictions[10][0]))


## 11. Ekspor hasil

Hasil benchmark dapat diunduh sebagai CSV untuk dimasukkan ke laporan.

In [ ]:
csv_output = "hasil_praktikum_6_micro_spotify.csv"
results_df.to_csv(csv_output, index=False)
print("CSV hasil tersimpan:", csv_output)

try:
    from google.colab import files
    files.download(csv_output)
except Exception:
    print("Unduh file CSV dari panel file runtime.")


# Analisis dan pembahasan kode

## A. Pemuatan dataset dan pemilihan fitur
Kode memuat CSV Micro Spotify, kemudian mengambil sembilan fitur numerik yang menggambarkan karakteristik audio lagu. Pemilihan fitur ini membuat pencarian didasarkan pada karakteristik audio, bukan judul, artis, genre, atau lirik. Karena itu, dua lagu yang berdekatan dalam ruang fitur belum tentu memiliki genre atau makna yang sama.

## B. StandardScaler
StandardScaler mengubah fitur menggunakan rumus \(z=(x-\mu)/\sigma\). Standardisasi mencegah fitur berskala besar seperti tempo atau loudness mendominasi jarak Euclidean hanya karena rentang numeriknya. Pada dataset dengan nilai konstan, simpangan baku ditangani oleh implementasi StandardScaler.

## C. Exact Nearest Neighbors
Exact NN menghitung jarak ke seluruh data dan dipakai sebagai ground truth. Metode ini menjadi acuan akurasi, tetapi biaya komputasinya meningkat seiring bertambahnya jumlah lagu dan jumlah query. Notebook membatasi jumlah query evaluasi agar praktikum dapat dijalankan pada runtime Colab, sambil tetap membangun indeks ANN menggunakan semua baris valid.

## D. ANNOY
ANNOY menggunakan random projection trees. `n_trees` memengaruhi struktur indeks dan biaya build, sedangkan `search_k` mengendalikan jumlah kandidat yang diperiksa saat query. Nilai `search_k` yang lebih besar umumnya dapat meningkatkan recall, tetapi pencarian bisa lebih lambat.

## E. HNSWLIB
HNSWLIB membangun graf navigasi bertingkat. Parameter `M` mengendalikan jumlah koneksi graf, `ef_construction` memengaruhi kualitas konstruksi indeks, dan `ef` mengatur luas pencarian saat query. Menaikkan `ef` biasanya meningkatkan recall dengan konsekuensi waktu query yang lebih tinggi.

## F. FAISS IVF-Flat
FAISS IVF-Flat mengelompokkan vektor ke dalam inverted lists. `nlist` adalah jumlah cluster, sedangkan `nprobe` menentukan banyaknya cluster yang diperiksa per query. `nprobe` yang lebih tinggi umumnya meningkatkan recall, tetapi juga meningkatkan jumlah perhitungan jarak. IVF-Flat menyimpan vektor penuh sehingga berbeda dari IVF-PQ yang melakukan kompresi.

## G. Recall@10
Recall@10 dihitung sebagai jumlah tetangga yang sama antara hasil ANN dan hasil exact, dibagi jumlah seluruh tetangga exact yang dievaluasi. Nilai mendekati 1 menunjukkan hasil ANN banyak cocok dengan ground truth. Karena query berasal dari dataset itu sendiri, notebook mengecualikan ID query dari hasil agar tidak selalu menjadi tetangga pertama.

## H. Perbandingan waktu
- **Build time** mengukur biaya awal pembuatan indeks.
- **Query time** mengukur waktu pencarian untuk seluruh query evaluasi.
- **QPS** dihitung sebagai jumlah query dibagi waktu query; nilai lebih tinggi berarti lebih banyak query diproses per detik.

Waktu total pada kode contoh awal mencampurkan build dan query sehingga sulit menilai performa pencarian murni. Notebook ini mencatat keduanya secara terpisah.

## I. Batasan dan kehati-hatian
1. Hasil aktual bergantung pada CPU, RAM, versi library, dan kondisi runtime.
2. Data yang tidak valid pada fitur yang dipilih dibuang; jumlahnya dicetak saat pemuatan.
3. `row index` pada hasil adalah indeks internal setelah pembersihan, bukan Spotify track ID.
4. FAISS IVF memerlukan training sebelum penambahan data; waktu training dimasukkan ke build time.
5. Exact NN menggunakan perhitungan jarak batch dan dapat menjadi bagian termahal. Jika memori atau waktu terbatas, turunkan `N_QUERY`.
6. Tidak ada angka hasil yang ditulis secara manual; tabel dan grafik dibentuk dari pengukuran aktual saat notebook dijalankan.

## Kesimpulan yang diharapkan
Praktikum ini menunjukkan bahwa ANNOY, FAISS IVF-Flat, dan HNSWLIB mempercepat pencarian dengan strategi indeks yang berbeda, sementara Exact NN menjadi pembanding akurasi. Metode terbaik harus dipilih berdasarkan trade-off recall, waktu query, waktu build, dan keterbatasan memori yang terukur pada dataset Micro Spotify. Kesimpulan akhir perlu ditulis berdasarkan tabel dan grafik hasil eksekusi Anda, bukan asumsi sebelum eksperimen dijalankan.
